学習済みLLMを使って「次のトークンの候補と確率を表示するコード」
日本語の文章生成モデル rinna/japanese-gpt2-medium を使う

In [ ]:
!pip -q install transformers sentencepiece

rinna/japanese-gpt2-mediumをロードする

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "rinna/japanese-gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
tokenizer.do_lower_case = True

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: rinna/japanese-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(32000, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=32000, bias=False)
)

② 入力文から、次のトークンを予測する

In [ ]:
# 入力する文章
text = "日本の首都は"

# 文章をトークンのIDに変換する
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

# モデルで予測する（重みの学習はしない）
with torch.no_grad():
    outputs = model(**inputs)

# 最後の位置の出力が「次のトークン」のスコア
scores = outputs.logits[0, -1]

# スコアを確率に変換する
probabilities = torch.softmax(scores, dim=-1)

# 確率が高い上位5候補を表示する
values, ids = torch.topk(probabilities, k=5)

for probability, token_id in zip(values, ids):
    token = tokenizer.decode([token_id.item()])
    print(f"{token!r}:{probability.item():.1%}")

'東京':18.0%
'、':10.0%
'ロンドン':3.8%
'北京':2.8%
'東京都':2.7%


In [ ]:
# 入力する文章

text = """文章：ネコがベッドに乗った。ネコはそこで眠った。
質問：この文章のそことは、どこですか？
答え："""

# 文章をトークンのIDに変換する
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

# モデルで予測する（重みの学習はしない）
with torch.no_grad():
    outputs = model(**inputs)

# 最後の位置の出力が「次のトークン」のスコア
scores = outputs.logits[0, -1]

# スコアを確率に変換する
probabilities = torch.softmax(scores, dim=-1)

# 確率が高い上位5候補を表示する
values, ids = torch.topk(probabilities, k=5)

for probability, token_id in zip(values, ids):
    token = tokenizer.decode([token_id.item()])
    print(f"{token!r}:{probability.item():.1%}")

'ベッド':11.1%
'「':6.5%
'この':4.5%
'':3.4%
'ネコ':2.5%
